In [9]:
# ========== 导入：QLoRA 微调 + 评估要用到的库 ==========
# os：读/写环境变量（如 WANDB_PROJECT）
import os
# PyTorch：张量、CUDA、训练后端
import torch
# Weights & Biases：记录 loss / 推送 checkpoint
import wandb
# 时间戳：拼到 run 名里，避免覆盖
from datetime import datetime
# Hugging Face datasets：加载定价语料
from datasets import load_dataset
# 因果 LM、分词器、4bit 配置、早停、固定种子
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
# LoRA 配置 + 从 Hub 加载已训适配器
from peft import LoraConfig, PeftModel
# TRL：SFTTrainer / SFTConfig；Completion-only collator（只对答案算 loss）
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
# math：评估里算 RMSLE
import math
# matplotlib：散点图可视化误差
import matplotlib.pyplot as plt
# F.softmax：Top-K 推理时把 logits 变概率
import torch.nn.functional as F


In [ ]:
# ========== 加载精简定价数据，再切出 train / val ==========
# 在我创建的精简定价数据上进行测试
# Hub 数据集 id（作者自建 lite pricer 数据）
DATASET_NAME = f"qshaikh/lite-pricer-data"
# 拉取完整 DatasetDict
dataset = load_dataset(DATASET_NAME)
# 原始 train / test split
train = dataset['train']
test = dataset['test']
# 从 train 再划 10% 做验证集
split_ratio = 0.10  # 10% for validation

# 先截断训练集规模，控制训练时长
TRAIN_SIZE = 15000
train = train.select(range(TRAIN_SIZE))

# 截断后的总条数
total_size = len(train)
# 验证集条数 = 总数 × 比例
val_size = int(total_size * split_ratio)

# 前 val_size 条 → 验证
val_data = train.select(range(val_size))
# 剩余 → 真正用于 SFT 的训练数据
train_data = train.select(range(val_size, total_size))


In [ ]:
# ========== 打印三个 split 的规模，确认切分无误 ==========
print(f"Train data size     : {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Test data size      : {len(test)}")


In [4]:
# ========== 命名本次实验：项目名 / run 名 / Hub 模型名 ==========
# W&B 与本地输出目录用的项目名
PROJECT_NAME = "llama3-new-pricer"
# run 名：时间戳 + 训练规模，保证唯一
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}-size{total_size}"
# 本地 output_dir / 推 Hub 时常用的完整名字
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# 预期推到 Hub 的仓库路径：{user}/{project-run}
HUB_MODEL_NAME = f"qshaikh/{PROJECT_RUN_NAME}"


In [ ]:
# ========== 配置并初始化 Weights & Biases ==========
# 是否把训练过程打到 W&B
LOG_TO_WANDB = True
# 项目名写入环境变量，供 transformers/trl 上报
os.environ["WANDB_PROJECT"] = PROJECT_NAME
# 日志策略：开 W&B 时记 checkpoint，否则只在结束记
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
# 记录梯度曲线
os.environ["WANDB_WATCH"] = "gradients"

# 真正创建 W&B run（名称用上面的 RUN_NAME）
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)


In [ ]:
# ========== 4bit 加载 Llama 3.2-1B 基座 ==========
# 基座模型 id（1B，比 3B 更省显存）
BASE_MODEL = "meta-llama/Llama-3.2-1B"

# NF4 4bit + 双重量化；计算用 bfloat16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,  
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# 加载分词器；信任远程代码（部分模型需要）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# 用 eos 充当 pad，避免缺 pad_token
tokenizer.pad_token = tokenizer.eos_token
# 右填充，适配因果语言模型
tokenizer.padding_side = "right"

# 4bit 加载基座到自动设备映射
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置补上 pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 打印显存占用（MB），确认量化生效
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


In [7]:
# ========== Completion-only collator：只对「答案段」算损失 ==========
# 模板字符串：提示里价格答案从这里开始（必须与数据格式一致）
response_template = "Price is $"
# DataCollatorForCompletionOnlyLM：把 template 之前的 label 置忽略，避免背诵问题
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)


In [8]:
# ========== LoRA 超参：秩、缩放、作用模块、dropout ==========
# 低秩维度 r：越大表达力越强、参数也越多
LORA_R = 32
# alpha：LoRA 缩放，常取 2*r
LORA_ALPHA = 64
# 注入注意力的 q/k/v/o 投影
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
# LoRA 分支 dropout，减轻过拟合
LORA_DROPOUT = 0.1


In [9]:
# ========== 组装 peft.LoraConfig ==========
lora_parameters = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    # 不训练 bias
    bias="none",
    # 因果语言模型任务类型
    task_type="CAUSAL_LM"
)


In [ ]:
# ========== 训练超参：batch、学习率、保存节奏等 ==========
# 训练轮数
EPOCHS = 1
# 每卡 batch；作者从 4 试到 8 以加快（注释保留英文原意）
BATCH_SIZE = 8          # Tested it with 4 first to be on the safer side, however was taking too long. Increased it upto 8 then.
# 梯度累积步数（等效更大 batch）
GRADIENT_ACCUMULATION_STEPS = 1
# 截断/填充的最大序列长度
MAX_SEQUENCE_LENGTH = 182
# 学习率
LEARNING_RATE = 1e-4
# 余弦退火调度
LR_SCHEDULER_TYPE = 'cosine'
# 预热占总步数比例
WARMUP_RATIO = 0.03
# 分页 AdamW，4bit 训练常用
OPTIMIZER = "paged_adamw_32bit"

# 每多少 step 存盘
SAVE_STEPS = 200
# 日志 / 评估间隔
STEPS = 20
# 最多保留多少个 checkpoint
save_total_limit = 10


In [ ]:
# ========== SFTConfig：把上面的超参喂给 TRL ==========
train_parameters = SFTConfig(
    # 本地输出目录名
    output_dir=PROJECT_RUN_NAME,
    # W&B / 日志里的 run 名
    run_name=RUN_NAME,
    # 数据集里文本字段名
    dataset_text_field="text",
    # 最大序列长度
    max_seq_length=MAX_SEQUENCE_LENGTH,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    # -1 表示按 epoch 跑满，不用 max_steps 截断
    max_steps=-1,
    # 按长度分桶，减少 padding 浪费
    group_by_length=True,

    # 按 step 做验证
    eval_strategy="steps",
    eval_steps=STEPS,
    # 验证 batch 取 1，省显存
    per_device_eval_batch_size=1,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIMIZER,
    weight_decay=0.001,
    max_grad_norm=0.3,

    # 用 bf16，不用 fp16
    fp16=False,
    bf16=True,

    logging_steps=STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=save_total_limit,
    # 是否上报 W&B
    report_to="wandb" if LOG_TO_WANDB else None,

    # 训练过程推送到 Hub
    push_to_hub=True,
    hub_strategy="every_save",
    # 结束时加载验证集上最好的权重
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # loss 越小越好
    greater_is_better=False
)


In [ ]:
# ========== 构造 SFTTrainer：模型 + 数据 + LoRA + 早停 ==========
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_parameters,    
    args=train_parameters,          
    data_collator=collator,
    # 验证指标连续 5 次不提升就停
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
)


In [ ]:
# ========== 启动训练；结束后提示 Hub 模型名 ==========
fine_tuning.train()
print(f"Best model pushed to HF Hub: {HUB_MODEL_NAME}")


## 评估模型性能

下面自定义一个简易 `Tester`：对测试集逐条调用预测函数，算绝对误差与 RMSLE，并用颜色区分好/中/差，画「真值 vs 预测」散点图。


In [ ]:
# ========== 终端颜色 + Tester 评估类 ==========
# ANSI 颜色码：绿 / 黄 / 红 / 复位
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
# 颜色名 → ANSI，供需要彩色打印时使用
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}

class Tester:

    def __init__(self, predictor, data, title=None, size=250):
        # 预测函数：输入 text，输出价格 float
        self.predictor = predictor
        self.data = data
        # 图标题：默认用函数名美化
        self.title = title or predictor.__name__.replace("_", " ").title()
        # 评估条数
        self.size = size
        # 累积容器
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # 误差小或相对误差 <20% → 绿
        if error<40 or error/truth < 0.2:
            return "green"
        # 中等误差 → 橙
        elif error<80 or error/truth < 0.4:
            return "orange"
        # 否则红
        else:
            return "red"

    def run_datapoint(self, i):
        # 取第 i 条
        datapoint = self.data[i]
        # 模型猜测
        guess = self.predictor(datapoint["text"])
        # 真值价格
        truth = datapoint["price"]
        # 绝对误差
        error = abs(guess - truth)
        # 对数误差，用于 RMSLE
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        # 记入列表
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)

    def chart(self, title):
        # 画真值-预测散点 + y=x 参考线
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)

        from matplotlib.lines import Line2D
        # 图例：三种误差档
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Accurate (green)', markerfacecolor='green', markersize=8),
            Line2D([0], [0], marker='o', color='w', label='Medium error (orange)', markerfacecolor='orange', markersize=8),
            Line2D([0], [0], marker='o', color='w', label='High error (red)', markerfacecolor='red', markersize=8)
        ]
        plt.legend(handles=legend_elements, loc='upper right')

        plt.show()


    def report(self):
        # 平均绝对误差
        average_error = sum(self.errors) / self.size
        # 均方根对数误差
        rmsle = math.sqrt(sum(self.sles) / self.size)
        # 绿色点占比
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        # 逐条评估
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function, data):
        # 便捷入口：直接跑默认 size
        cls(function, data).run()


In [ ]:
# ========== 查看测试集第一条，确认字段（text / price 等） ==========
test[0]


In [ ]:
# ========== 从 Hub 加载已训好的 QLoRA 适配器 ==========
# 固定指向作者已推送的最佳 run（不必等于本次刚训的 HUB_MODEL_NAME）
FINETUNED_MODEL = "qshaikh/llama3-new-pricer-2025-10-29_00.32.54-size15000"
# 挂到当前 4bit 基座上
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")
# 笔记本里展示模型对象摘要
fine_tuned_model


In [ ]:
# ========== Top-K 概率混合推理：看下一 token 的价格分布 ==========
# 取前 K 个候选 token
top_K = 3

def improved_model_predict(prompt, device="cuda"):
    # 固定种子，保证同 prompt 可复现
    set_seed(42) 
    # 编码为 token id 张量并放到指定设备
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    # 全 1 attention mask（无 padding）
    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad(): 
        # 前向：拿最后一个位置的 next-token logits
        outputs = fine_tuned_model(inputs, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :].to('cpu')

    # logits → 概率
    next_token_probs = F.softmax(next_token_logits, dim=-1)
    # Top-K 概率与 token id
    top_prob, top_token_id = next_token_probs.topk(top_K)

    prices, weights = [], [] 

    for i in range(top_K):
      # 解码候选 token 文本
      predicted_token = tokenizer.decode(top_token_id[0][i])
      probability = top_prob[0][i]

      try:
        # 能解析成数字才保留
        result = float(predicted_token)
      except ValueError as e:
        result = 0.0

      if result > 0:
        prices.append(result)
        weights.append(probability)

    # 没有任何有效价格
    if not prices:
      return 0.0, 0.0

    # 归一化权重
    total = sum(weights)

    # 加权价格列表
    weighted_prices = [price * weight / total for price, weight in zip(prices, weights)]

    # 返回标量加权和（注意：这里只 return 一个 float；与「失败时返回二元组」签名不一致，但保持原逻辑）
    return sum(weighted_prices).item()


In [ ]:
# ========== 单条冒烟：对 test[0] 的 text 做一次 Top-K 预测 ==========
improved_model_predict(test[0]["text"], device="cuda")


In [ ]:
# ========== 在测试集上跑完整 Tester 报告 ==========
Tester.test(improved_model_predict, test)
